# Punto 3: A* y BFS en el 8-puzzle

Comparamos tres estrategias sobre el mismo tablero:

1. BFS
2. A* con fichas fuera de lugar
3. A* con distancia Manhattan

El estado es una tupla de 9 casillas. El `0` es el espacio vacío. `expandidos` es el número de estados que salen de la frontera. El tablero inicial está a 31 movimientos del objetivo, la distancia máxima del 8-puzzle.

In [ ]:
from collections import deque
from heapq import heappush, heappop
from itertools import count
from time import time
import math

In [ ]:
# Caso con solución: este tablero está a 31 movimientos, la distancia máxima del 8-puzzle.
estado_inicial = (
    8, 6, 7,
    2, 5, 4,
    3, 0, 1,
)

# Configuración que queremos alcanzar.
estado_objetivo = (
    1, 2, 3,
    4, 5, 6,
    7, 8, 0,
)

estado_inicial, estado_objetivo

((8, 6, 7, 2, 5, 4, 3, 0, 1), (1, 2, 3, 4, 5, 6, 7, 8, 0))

In [ ]:
def sucesores_8_puzzle(estado):
    # Convertimos la tupla a lista porque necesitamos intercambiar posiciones.
    estado = list(estado)

    # Buscamos dónde está la casilla vacía.
    indice_cero = estado.index(0)

    # Convertimos el índice lineal 0..8 en fila y columna de una matriz 3x3.
    fila, columna = divmod(indice_cero, 3)

    # Posibles desplazamientos de la casilla vacía:
    # arriba, abajo, izquierda y derecha.
    movimientos = [
        (-1, 0),
        (1, 0),
        (0, -1),
        (0, 1),
    ]

    sucesores = []

    for df, dc in movimientos:
        nf, nc = fila + df, columna + dc

        if 0 <= nf < 3 and 0 <= nc < 3:
            nuevo_indice = nf * 3 + nc
            nuevo = estado.copy()
            nuevo[indice_cero], nuevo[nuevo_indice] = (
                nuevo[nuevo_indice],
                nuevo[indice_cero],
            )
            sucesores.append(tuple(nuevo))

    return sucesores


sucesores_8_puzzle(estado_inicial)

[(8, 6, 7, 2, 0, 4, 3, 5, 1),
 (8, 6, 7, 2, 5, 4, 0, 3, 1),
 (8, 6, 7, 2, 5, 4, 3, 1, 0)]

In [ ]:
def fichas_fuera_de_lugar(estado, objetivo):
    # Contamos las fichas que no están en su posición, sin incluir el vacío.
    fuera = 0
    for valor_estado, valor_objetivo in zip(estado, objetivo):
        if valor_estado != 0 and valor_estado != valor_objetivo:
            fuera += 1
    return fuera


def manhattan_8_puzzle(estado, objetivo):
    # Posición objetivo de cada ficha, ignorando el vacío.
    posicion_objetivo = {ficha: indice for indice, ficha in enumerate(objetivo)}
    distancia = 0

    for indice, ficha in enumerate(estado):
        if ficha == 0:
            continue
        destino = posicion_objetivo[ficha]
        fila_actual, col_actual = divmod(indice, 3)
        fila_destino, col_destino = divmod(destino, 3)
        distancia += abs(fila_actual - fila_destino) + abs(col_actual - col_destino)

    return distancia

In [ ]:
def reconstruir_camino(padre, nodo):
    estados = []
    actual = nodo
    while actual is not None:
        estados.append(actual)
        actual = padre[actual]
    estados.reverse()
    return estados


def bfs_8_puzzle(estado_inicial, estado_objetivo):
    """
    Búsqueda primero en anchura para el 8-puzzle.
    No usa heurística. expandidos cuenta los estados que salen de la cola.
    """
    estado_inicial = tuple(estado_inicial)
    estado_objetivo = tuple(estado_objetivo)
    visitados = {estado_inicial}
    cola = deque([estado_inicial])
    padre = {estado_inicial: None}
    expandidos = 0

    while cola:
        actual = cola.popleft()
        expandidos += 1

        if actual == estado_objetivo:
            camino = reconstruir_camino(padre, actual)
            return {
                "camino": camino,
                "movimientos": len(camino) - 1,
                "expandidos": expandidos,
            }

        for sucesor in sucesores_8_puzzle(actual):
            if sucesor not in visitados:
                visitados.add(sucesor)
                padre[sucesor] = actual
                cola.append(sucesor)

    return {
        "camino": None,
        "movimientos": None,
        "expandidos": expandidos,
    }

In [ ]:
def reconstruir_camino(padre, nodo):
    estados = []
    actual = nodo
    while actual is not None:
        estados.append(actual)
        actual = padre[actual]
    estados.reverse()
    return estados


def astar_8_puzzle(estado_inicial, estado_objetivo, heuristica):
    """
    A* para el 8-puzzle. heuristica(estado, objetivo) estima los movimientos restantes.
    expandidos cuenta los estados que salen de la frontera.
    """
    estado_inicial = tuple(estado_inicial)
    estado_objetivo = tuple(estado_objetivo)

    desempate = count()
    frontera = []
    heappush(
        frontera,
        (heuristica(estado_inicial, estado_objetivo), 0, next(desempate), estado_inicial),
    )

    mejor_g = {estado_inicial: 0}
    padre = {estado_inicial: None}
    expandidos = 0

    while frontera:
        f, g, _, nodo = heappop(frontera)

        if g != mejor_g.get(nodo):
            continue

        expandidos += 1

        if nodo == estado_objetivo:
            camino = reconstruir_camino(padre, nodo)
            return {
                "camino": camino,
                "movimientos": len(camino) - 1,
                "expandidos": expandidos,
            }

        for sucesor in sucesores_8_puzzle(nodo):
            nuevo_g = g + 1
            if nuevo_g < mejor_g.get(sucesor, math.inf):
                mejor_g[sucesor] = nuevo_g
                padre[sucesor] = nodo
                nuevo_f = nuevo_g + heuristica(sucesor, estado_objetivo)
                heappush(
                    frontera,
                    (nuevo_f, nuevo_g, next(desempate), sucesor),
                )

    return {
        "camino": None,
        "movimientos": None,
        "expandidos": expandidos,
    }

In [ ]:
def medir(funcion):
    t0 = time()
    resultado = funcion()
    resultado["tiempo"] = time() - t0
    return resultado


resultado_bfs = medir(lambda: bfs_8_puzzle(estado_inicial, estado_objetivo))
resultado_h1 = medir(
    lambda: astar_8_puzzle(estado_inicial, estado_objetivo, fichas_fuera_de_lugar)
)
resultado_h2 = medir(
    lambda: astar_8_puzzle(estado_inicial, estado_objetivo, manhattan_8_puzzle)
)

{
    "BFS": {
        "movimientos": resultado_bfs["movimientos"],
        "expandidos": resultado_bfs["expandidos"],
        "tiempo": resultado_bfs["tiempo"],
    },
    "A* fichas fuera de lugar": {
        "movimientos": resultado_h1["movimientos"],
        "expandidos": resultado_h1["expandidos"],
        "tiempo": resultado_h1["tiempo"],
    },
    "A* Manhattan": {
        "movimientos": resultado_h2["movimientos"],
        "expandidos": resultado_h2["expandidos"],
        "tiempo": resultado_h2["tiempo"],
    },
}

{'BFS': {'movimientos': 31,
  'expandidos': 181439,
  'tiempo': 0.40037083625793457},
 'A* fichas fuera de lugar': {'movimientos': 31,
  'expandidos': 143849,
  'tiempo': 0.6795558929443359},
 'A* Manhattan': {'movimientos': 31,
  'expandidos': 21198,
  'tiempo': 0.16448092460632324}}